# ELT - Integração de Dados para Previsão de Evasão Escolar

Este notebook realiza o processo de **ELT (Extract, Load, Transform)** para integrar:

- Microdados do Censo Escolar
- Dados socioeconômicos municipais

Objetivo final:

Criar datasets analíticos que serão utilizados para **modelagem de machine learning para previsão de evasão escolar**.

Estrutura do pipeline:

RAW → SILVER → GOLD

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [3]:
BASE_DIR = Path("data")

RAW = BASE_DIR / "raw"
SILVER = BASE_DIR / "silver"
GOLD = BASE_DIR / "gold"

MICRODADOS = RAW / "microdados_inep"
SOCIO = RAW / "socioeconomicos"

SILVER.mkdir(parents=True, exist_ok=True)
GOLD.mkdir(parents=True, exist_ok=True)

In [4]:
PATH_MATRICULA = MICRODADOS / "Tabela_Matricula_2025.csv"
PATH_ESCOLA = MICRODADOS / "Tabela_Escola_2025.csv"
PATH_TURMA = MICRODADOS / "Tabela_Turma_2025.csv"
PATH_DOCENTE = MICRODADOS / "Tabela_Docente_2025.csv"
PATH_GESTOR = MICRODADOS / "Tabela_Gestor_Escolar_2025.csv"

PATH_MUNICIPIOS = SOCIO / "base_municipios_brasileiros.csv"
PATH_SOCIO_EXTRA = SOCIO / "municipios.csv"

## Escola enriquecida

In [5]:
def load_csv(path):
    return pd.read_csv(
        path,
        sep=";",
        encoding="latin1",
        low_memory=False
    )

print("Carregando microdados...")

matricula = load_csv(PATH_MATRICULA)
escola = load_csv(PATH_ESCOLA)
turma = load_csv(PATH_TURMA)
docente = load_csv(PATH_DOCENTE)
gestor = load_csv(PATH_GESTOR)

print("Microdados carregados!")

Carregando microdados...
Microdados carregados!


In [6]:
def load_csv(path):
    return pd.read_csv(
        path,
        sep=",",
        encoding="latin1",
        low_memory=False
    )

print("Carregando municipios...")

municipios = load_csv(PATH_MUNICIPIOS)

print("Municipios carregados!")

Carregando municipios...
Municipios carregados!


In [7]:
print("Formato do dataset:")

print("matricula:", matricula.shape)
print("escola:", escola.shape)
print("turma:", turma.shape)
print("docente:", docente.shape)
print("gestor:", gestor.shape)
print("municipios:", municipios.shape)

Formato do dataset:
matricula: (178766, 237)
escola: (214192, 302)
turma: (178772, 190)
docente: (178772, 156)
gestor: (180540, 65)
municipios: (180285, 452)


In [8]:
print("Colunas da tabela escola")
print(escola.columns.tolist())

print("\nColunas da tabela municipios")
print(municipios.columns.tolist())

Colunas da tabela escola
['NU_ANO_CENSO', 'NO_REGIAO', 'CO_REGIAO', 'NO_UF', 'SG_UF', 'CO_UF', 'NO_MUNICIPIO', 'CO_MUNICIPIO', 'NO_REGIAO_GEOG_INTERM', 'CO_REGIAO_GEOG_INTERM', 'NO_REGIAO_GEOG_IMED', 'CO_REGIAO_GEOG_IMED', 'NO_MESORREGIAO', 'CO_MESORREGIAO', 'NO_MICRORREGIAO', 'CO_MICRORREGIAO', 'NO_DISTRITO', 'CO_DISTRITO', 'NO_REGIAO_ADMINISTRATIVA', 'CO_REGIAO_ADMINISTRATIVA', 'NO_ENTIDADE', 'CO_ENTIDADE', 'TP_DEPENDENCIA', 'TP_CATEGORIA_ESCOLA_PRIVADA', 'TP_LOCALIZACAO', 'TP_LOCALIZACAO_DIFERENCIADA', 'DS_ENDERECO', 'NU_ENDERECO', 'DS_COMPLEMENTO', 'NO_BAIRRO', 'CO_CEP', 'NU_DDD', 'NU_TELEFONE', 'LATITUDE', 'LONGITUDE', 'TP_SITUACAO_FUNCIONAMENTO', 'CO_ORGAO_REGIONAL', 'DT_ANO_LETIVO_INICIO', 'DT_ANO_LETIVO_TERMINO', 'IN_VINCULO_SECRETARIA_EDUCACAO', 'IN_VINCULO_SEGURANCA_PUBLICA', 'IN_VINCULO_SECRETARIA_SAUDE', 'IN_VINCULO_OUTRO_ORGAO', 'IN_PODER_PUBLICO_PARCERIA', 'TP_PODER_PUBLICO_PARCERIA', 'IN_FORMA_CONT_TERMO_COLABORA', 'IN_FORMA_CONT_TERMO_FOMENTO', 'IN_FORMA_CONT_ACORDO_COO

In [9]:
escola.head()

,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,NO_REGIAO_GEOG_INTERM,CO_REGIAO_GEOG_INTERM,NO_REGIAO_GEOG_IMED,CO_REGIAO_GEOG_IMED,NO_MESORREGIAO,CO_MESORREGIAO,NO_MICRORREGIAO,CO_MICRORREGIAO,NO_DISTRITO,CO_DISTRITO,NO_REGIAO_ADMINISTRATIVA,CO_REGIAO_ADMINISTRATIVA,NO_ENTIDADE,CO_ENTIDADE,TP_DEPENDENCIA,TP_CATEGORIA_ESCOLA_PRIVADA,TP_LOCALIZACAO,TP_LOCALIZACAO_DIFERENCIADA,DS_ENDERECO,NU_ENDERECO,DS_COMPLEMENTO,NO_BAIRRO,CO_CEP,NU_DDD,NU_TELEFONE,LATITUDE,LONGITUDE,TP_SITUACAO_FUNCIONAMENTO,CO_ORGAO_REGIONAL,DT_ANO_LETIVO_INICIO,DT_ANO_LETIVO_TERMINO,IN_VINCULO_SECRETARIA_EDUCACAO,IN_VINCULO_SEGURANCA_PUBLICA,IN_VINCULO_SECRETARIA_SAUDE,IN_VINCULO_OUTRO_ORGAO,IN_PODER_PUBLICO_PARCERIA,TP_PODER_PUBLICO_PARCERIA,IN_FORMA_CONT_TERMO_COLABORA,IN_FORMA_CONT_TERMO_FOMENTO,IN_FORMA_CONT_ACORDO_COOP,IN_FORMA_CONT_PRESTACAO_SERV,IN_FORMA_CONT_COOP_TEC_FIN,IN_FORMA_CONT_CONSORCIO_PUB,IN_FORMA_CONT_MU_TERMO_COLAB,IN_FORMA_CONT_MU_TERMO_FOMENTO,IN_FORMA_CONT_MU_ACORDO_COOP,IN_FORMA_CONT_MU_PREST_SERV,IN_FORMA_CONT_MU_COOP_TEC_FIN,IN_FORMA_CONT_MU_CONSORCIO_PUB,IN_FORMA_CONT_ES_TERMO_COLAB,IN_FORMA_CONT_ES_TERMO_FOMENTO,IN_FORMA_CONT_ES_ACORDO_COOP,IN_FORMA_CONT_ES_PREST_SERV,IN_FORMA_CONT_ES_COOP_TEC_FIN,IN_FORMA_CONT_ES_CONSORCIO_PUB,IN_MANT_ESCOLA_PRIVADA_EMP,IN_MANT_ESCOLA_PRIVADA_ONG,IN_MANT_ESCOLA_PRIVADA_OSCIP,IN_MANT_ESCOLA_PRIV_ONG_OSCIP,IN_MANT_ESCOLA_PRIVADA_SIND,IN_MANT_ESCOLA_PRIVADA_SIST_S,IN_MANT_ESCOLA_PRIVADA_S_FINS,NU_CNPJ_ESCOLA_PRIVADA,NU_CNPJ_MANTENEDORA,TP_REGULAMENTACAO,TP_RESPONSAVEL_REGULAMENTACAO,CO_ESCOLA_SEDE_VINCULADA,CO_IES_OFERTANTE,IN_LOCAL_FUNC_PREDIO_ESCOLAR,TP_OCUPACAO_PREDIO_ESCOLAR,IN_LOCAL_FUNC_SOCIOEDUCATIVO,IN_LOCAL_FUNC_UNID_PRISIONAL,IN_LOCAL_FUNC_PRISIONAL_SOCIO,IN_LOCAL_FUNC_GALPAO,TP_OCUPACAO_GALPAO,IN_LOCAL_FUNC_SALAS_OUTRA_ESC,IN_LOCAL_FUNC_OUTROS,IN_PREDIO_COMPARTILHADO,IN_AGUA_POTAVEL,IN_AGUA_REDE_PUBLICA,IN_AGUA_POCO_ARTESIANO,IN_AGUA_CACIMBA,IN_AGUA_FONTE_RIO,IN_AGUA_INEXISTENTE,IN_AGUA_CARRO_PIPA,IN_ENERGIA_REDE_PUBLICA,IN_ENERGIA_GERADOR_FOSSIL,IN_ENERGIA_RENOVAVEL,IN_ENERGIA_INEXISTENTE,IN_ESGOTO_REDE_PUBLICA,IN_ESGOTO_FOSSA_SEPTICA,IN_ESGOTO_FOSSA_COMUM,...,QT_PROF_BIBLIOTECARIO,QT_PROF_SAUDE,QT_PROF_COORDENADOR,QT_PROF_FONAUDIOLOGO,QT_PROF_NUTRICIONISTA,QT_PROF_PSICOLOGO,QT_PROF_ALIMENTACAO,QT_PROF_PEDAGOGIA,QT_PROF_SECRETARIO,QT_PROF_SEGURANCA,QT_PROF_MONITORES,QT_PROF_GESTAO,QT_PROF_ASSIST_SOCIAL,QT_PROF_TRAD_LIBRAS,QT_PROF_AGRICOLA,QT_PROF_REVISOR_BRAILLE,IN_ALIMENTACAO,IN_MATERIAL_PED_MULTIMIDIA,IN_MATERIAL_PED_INFANTIL,IN_MATERIAL_PED_CIENTIFICO,IN_MATERIAL_PED_DIFUSAO,IN_MATERIAL_PED_MUSICAL,IN_MATERIAL_PED_JOGOS,IN_MATERIAL_PED_ARTISTICAS,IN_MATERIAL_PED_PROFISSIONAL,IN_MATERIAL_PED_DESPORTIVA,IN_MATERIAL_PED_INDIGENA,IN_MATERIAL_PED_ETNICO,IN_MATERIAL_PED_CAMPO,IN_MATERIAL_PED_BIL_SURDOS,IN_MATERIAL_PED_AGRICOLA,IN_MATERIAL_PED_QUILOMBOLA,IN_MATERIAL_PED_EDU_ESP,IN_MATERIAL_PED_NENHUM,IN_EDUCACAO_INDIGENA,TP_INDIGENA_LINGUA,CO_LINGUA_INDIGENA_1,CO_LINGUA_INDIGENA_2,CO_LINGUA_INDIGENA_3,IN_EXAME_SELECAO,IN_RESERVA_PPI,IN_RESERVA_RENDA,IN_RESERVA_PUBLICA,IN_RESERVA_PCD,IN_RESERVA_OUTROS,IN_RESERVA_NENHUMA,IN_REDES_SOCIAIS,IN_ESPACO_ATIVIDADE,IN_ESPACO_EQUIPAMENTO,IN_ORGAO_ASS_PAIS,IN_ORGAO_ASS_PAIS_MESTRES,IN_ORGAO_CONSELHO_ESCOLAR,IN_ORGAO_GREMIO_ESTUDANTIL,IN_ORGAO_OUTROS,IN_ORGAO_NENHUM,TP_PROPOSTA_PEDAGOGICA,IN_EDUC_AMBIENTAL,IN_EDUC_AMB_CONTEUDO,IN_EDUC_AMB_CURRICULAR,IN_EDUC_AMB_EIXO,IN_EDUC_AMB_EVENTOS,IN_EDUC_AMB_PROJETOS,IN_EDUC_AMB_NENHUMA,TP_AEE,TP_ATIVIDADE_COMPLEMENTAR,TP_ITINERARIO_FORMATIVO,IN_ITINERARIO_APROFUNDAMENTO,IN_ITINERARIO_TECN_PROF,IN_ESCOLARIZACAO,IN_MEDIACAO_PRESENCIAL,IN_MEDIACAO_SEMIPRESENCIAL,IN_MEDIACAO_EAD,IN_ESPECIAL_EXCLUSIVA,IN_REGULAR,IN_EJA,IN_PROFISSIONALIZANTE,IN_COMUM_CRECHE,IN_COMUM_PRE,IN_COMUM_FUND_AI,IN_COMUM_FUND_AF,IN_COMUM_MEDIO_MEDIO,IN_COMUM_MEDIO_INTEGRADO,IN_COMUM_MEDIO_FIC,IN_COMUM_MEDIO_NORMAL,IN_ESP_EXCLUSIVA_CRECHE,IN_ESP_EXCLUSIVA_PRE,IN_ESP_EXCLUSIVA_FUND_AI,IN_ESP_EXCLUSIVA_FUND_AF,IN_ESP_

In [10]:
municipios.head()

,Unnamed: 0,id_municipio,id_municipio_6,id_municipio_tse,id_municipio_rf,id_municipio_bcb,nome_municipio,capital_uf,id_comarca,id_regiao_saude,nome_regiao_saude,id_regiao_imediata,nome_regiao_imediata,id_regiao_intermediaria,nome_regiao_intermediaria,id_microrregiao,nome_microrregiao,id_mesorregiao,nome_mesorregiao,id_regiao_metropolitana,nome_regiao_metropolitana,ddd,id_uf,nome_uf,nome_regiao,amazonia_legal,centroide,dimensao_identificao,ano,populacao,prop_catolicos,prop_evangelicos_pentecostais,prop_evangelicos_missao,prop_espiritas,prop_matriz_africana,ano_censo,dimensao_populacao,total_desastres,total_pessoas_afetadas,total_danos_materiais,total_prejuizos_publicos,total_prejuizos_privados,total_desastres_climatologicos,total_pessoas_afetadas_climatologicos,total_danos_materiais_climatologicos,total_prejuizos_publicos_climatologicos,total_prejuizos_privados_climatologicos,total_desastres_hidrologicos,total_pessoas_afetadas_hidrologicos,total_danos_materiais_hidrologicos,total_prejuizos_publicos_hidrologicos,total_prejuizos_privados_hidrologicos,total_desastres_meteorologicos,total_pessoas_afetadas_meteorologicos,total_danos_materiais_meteorologicos,total_prejuizos_publicos_meteorologicos,total_prejuizos_privados_meteorologicos,total_desastres_outros,total_pessoas_afetadas_outros,total_danos_materiais_outros,total_prejuizos_publicos_outros,total_prejuizos_privados_outros,populacao_atendida_agua,populacao_atentida_esgoto,populacao_urbana,populacao_urbana_residente_agua,populacao_urbana_atendida_esgoto,extensao_rede_agua,extensao_rede_esgoto,quantidade_sede_municipal_agua,quantidade_sede_municipal_esgoto,quantidade_ligacao_total_agua,quantidade_ligacao_total_esgoto,quantidade_empregado,despesa_pessoal,arrecadacao_total,investimento_agua_municipio,investimento_agua_estado,investimento_agua_prestador,investimento_esgoto_municipio,investimento_esgoto_estado,investimento_esgoto_prestador,indice_atendimento_total_agua,indice_atendimento_esgoto_agua,id_municipio_nome,nm_uf,area_total_municipio,desmatado_total_municipio,desmatado_total_amazonia,desmatado_total_caatinga,desmatado_total_cerrado,desmatado_total_mata_atlantica,desmatado_total_pampa,desmatado_total_pantanal,proporcao_desmatado,lag_total_desmatado,taxa_crescimento_desmatamento_abs,id,capacidade_investimento_adaptacao,capacidade_investimento_adaptacao_classe,...,total_mortalidade_parda_feminino,total_mortalidade_branca_feminino,total_mortalidade_amarela_feminino,total_mortalidade_indigena_feminino,total_mortalidade_homicidio_branca,total_mortalidade_homicidio_preta,total_mortalidade_homicidio_parda,total_mortalidade_homicidio_amarela,total_mortalidade_homicidio_indigena,total_mortalidade_domicilio,total_mortalidade_homicidio_domicilio,total_mortalidade_alcool_domicilio,total_mortalidade_homicidio_feminino_domicilio,total_mortalidade_homicidio_masculino_domicilio,total_mortalidade_alcool_feminino_domicilio,total_mortalidade_alcool_masculino_domicilio,total_mortalidade_homicidio_branca_domicilio,total_mortalidade_homicidio_preta_domicilio,total_mortalidade_homicidio_parda_domicilio,total_mortalidade_homicidio_amarela_domicilio,total_mortalidade_homicidio_indigena_domicilio,quantidade_mortes_intervencao_policial_civil_fora_de_servico,quantidade_feminicidio,quantidade_mortes_intervencao_policial_militar_fora_de_servico,quantidade_furto_veiculos,quantidade_mortes_intervencao_policial_civil_em_servico,quantidade_estupro,quantidade_morte_policiais_civis_confronto_em_servico,quantidade_mortes_intervencao_policial_militar_em_servico,quantidade_mortes_policiais_confronto,quantidade_mortes_violentas_intencionais,quantidade_posse_uso_entorpecente,quantidade_morte_policiais_militares_fora_de_servico,quantidade_morte_policiais_civis_fora_de_servico,quantidade_latrocinio,quantidade_porte_ilegal_arma_de_fogo,quantidade_mortes_intervencao_policial,quantidade_roubo_furto_veiculos,quantidade_posse_ilegal_arma_de_fogo,quantidade_lesao_corporal_dolosa_violencia_domestica,quantidade_trafico_ento

In [11]:
cols_escola = [
    "CO_ENTIDADE",
    "CO_MUNICIPIO",
    "CO_UF"
]

escola_sel = escola[cols_escola].copy()

In [12]:
cols_docente = [
    "CO_ENTIDADE",
    "QT_DOC_BAS",
    "QT_DOC_BAS_FEM",
    "QT_DOC_BAS_MASC"
]

docente_sel = docente[cols_docente].copy()

In [13]:
cols_gestor = [
    "CO_ENTIDADE",
    "QT_GEST_BAS"
]

gestor_sel = gestor[cols_gestor].copy()

In [14]:
escola_full = escola_sel.merge(
    docente_sel,
    on="CO_ENTIDADE",
    how="left"
)

In [15]:
print(escola_full.shape)

escola_full.head()

(214192, 6)


,CO_ENTIDADE,CO_MUNICIPIO,CO_UF,QT_DOC_BAS,QT_DOC_BAS_FEM,QT_DOC_BAS_MASC
0,11022558,1100015,11,1.0,0.0,1.0
1,11024275,1100015,11,9.0,5.0,4.0
2,11024291,1100015,11,NaN,NaN,NaN
3,11024666,1100015,11,13.0,10.0,3.0
4,11024682,1100015,11,44.0,24.0,20.0


In [16]:
gestor.columns.tolist()

['NU_ANO_CENSO',
 'CO_ENTIDADE',
 'QT_GEST_BAS',
 'QT_GEST_BAS_FEM',
 'QT_GEST_BAS_MASC',
 'QT_GEST_BAS_ND',
 'QT_GEST_BAS_BRANCA',
 'QT_GEST_BAS_PRETA',
 'QT_GEST_BAS_PARDA',
 'QT_GEST_BAS_AMARELA',
 'QT_GEST_BAS_INDIGENA',
 'QT_GEST_BAS_NACIO_BRASILEIRA',
 'QT_GEST_BAS_NACIO_ESTRANG',
 'QT_GEST_BAS_0_24',
 'QT_GEST_BAS_25_29',
 'QT_GEST_BAS_30_39',
 'QT_GEST_BAS_40_49',
 'QT_GEST_BAS_50_54',
 'QT_GEST_BAS_55_59',
 'QT_GEST_BAS_60_MAIS',
 'QT_GEST_BAS_PCD',
 'QT_GEST_BAS_ZR_URB',
 'QT_GEST_BAS_ZR_RUR',
 'QT_GEST_BAS_ZR_NA',
 'QT_GEST_BAS_ESCO_EF',
 'QT_GEST_BAS_ESCO_EM',
 'QT_GEST_BAS_ESCO_SUP_GRAD',
 'QT_GEST_BAS_ESCO_SUP_GRAD_LICEN',
 'QT_GEST_BAS_ESCO_SUP_GRAD_SLICEN',
 'QT_GEST_BAS_ESCO_SUP_POS_ESPEC',
 'QT_GEST_BAS_ESCO_SUP_POS_MESTRA',
 'QT_GEST_BAS_ESCO_SUP_POS_DOUTO',
 'QT_GEST_BAS_ESCO_SUP_POS_NENHUM',
 'QT_GEST_BAS_VINCULO_CONCUR',
 'QT_GEST_BAS_VINCULO_CONTRA',
 'QT_GEST_BAS_VINCULO_TERCEIR',
 'QT_GEST_BAS_VINCULO_CLT',
 'QT_GEST_BAS_DIRETOR',
 'QT_GEST_BAS_OUTRO',
 'QT_GES

In [17]:
gestor_sel = (
    gestor
    .groupby("CO_ENTIDADE")
    .size()
    .reset_index(name="QT_GESTORES")
)

In [18]:
escola_full = escola_full.merge(
    gestor_sel,
    on="CO_ENTIDADE",
    how="left"
)

In [19]:
escola_full.head()

,CO_ENTIDADE,CO_MUNICIPIO,CO_UF,QT_DOC_BAS,QT_DOC_BAS_FEM,QT_DOC_BAS_MASC,QT_GESTORES
0,11022558,1100015,11,1.0,0.0,1.0,1.0
1,11024275,1100015,11,9.0,5.0,4.0,1.0
2,11024291,1100015,11,NaN,NaN,NaN,NaN
3,11024666,1100015,11,13.0,10.0,3.0,1.0
4,11024682,1100015,11,44.0,24.0,20.0,1.0


In [20]:
escola_full.to_parquet(
    SILVER / "escola_enriquecida.parquet",
    index=False
)

print("Tabela escola_enriquecida salva!")

Tabela escola_enriquecida salva!


## Turma enriquecida 

In [21]:
cols_turma = [
    "CO_ENTIDADE",
    "QT_TUR_BAS",
    "QT_TUR_INF",
    "QT_TUR_FUND",
    "QT_TUR_MED",
    "QT_TUR_EJA",
    "QT_TUR_PROF"
]

turma_sel = turma[cols_turma].copy()

Colunas da tabela turma:
['NU_ANO_CENSO', 'CO_ENTIDADE', 'QT_TUR_BAS', 'QT_TUR_INF', 'QT_TUR_INF_CRE', 'QT_TUR_INF_PRE', 'QT_TUR_FUND', 'QT_TUR_FUND_AI', 'QT_TUR_FUND_AI_1', 'QT_TUR_FUND_AI_2', 'QT_TUR_FUND_AI_3', 'QT_TUR_FUND_AI_4', 'QT_TUR_FUND_AI_5', 'QT_TUR_FUND_AI_MULTIETAPA', 'QT_TUR_FUND_AF', 'QT_TUR_FUND_AF_6', 'QT_TUR_FUND_AF_7', 'QT_TUR_FUND_AF_8', 'QT_TUR_FUND_AF_9', 'QT_TUR_FUND_AF_MULTI', 'QT_TUR_FUND_AF_CORRFLUXO', 'QT_TUR_MED', 'QT_TUR_MED_PROP', 'QT_TUR_MED_PROP_1', 'QT_TUR_MED_PROP_2', 'QT_TUR_MED_PROP_3', 'QT_TUR_MED_PROP_4', 'QT_TUR_MED_PROP_NS', 'QT_TUR_MED_IFTP_CT', 'QT_TUR_MED_IFTP_CT_1', 'QT_TUR_MED_IFTP_CT_2', 'QT_TUR_MED_IFTP_CT_3', 'QT_TUR_MED_IFTP_CT_4', 'QT_TUR_MED_IFTP_CT_NS', 'QT_TUR_MED_IFTP_QP', 'QT_TUR_MED_IFTP_QP_1', 'QT_TUR_MED_IFTP_QP_2', 'QT_TUR_MED_IFTP_QP_3', 'QT_TUR_MED_IFTP_QP_4', 'QT_TUR_MED_IFTP_QP_NS', 'QT_TUR_MED_NM', 'QT_TUR_MED_NM_1', 'QT_TUR_MED_NM_2', 'QT_TUR_MED_NM_3', 'QT_TUR_MED_NM_4', 'QT_TUR_MED_IFA_EXC', 'QT_TUR_MED_IFTP_EXC', 'QT_

In [25]:
cols_turma = [
    "CO_ENTIDADE",
    "QT_TUR_BAS",
    "QT_TUR_INF",
    "QT_TUR_FUND",
    "QT_TUR_MED",
    "QT_TUR_EJA",
    "QT_TUR_PROF"
]

turma_sel = turma[cols_turma].copy()

In [26]:
turma_sel.head()

,CO_ENTIDADE,QT_TUR_BAS,QT_TUR_INF,QT_TUR_FUND,QT_TUR_MED,QT_TUR_EJA,QT_TUR_PROF
0,11000023,15,0,15,0,0,0
1,11000040,12,12,0,0,0,0
2,11000058,35,3,23,9,0,4
3,11000104,25,5,20,0,0,0
4,11000198,45,11,28,6,0,0


In [27]:
escola_enriquecida = pd.read_parquet(
    SILVER / "escola_enriquecida.parquet"
)

escola_turma = escola_enriquecida.merge(
    turma_sel,
    on="CO_ENTIDADE",
    how="left"
)

In [28]:
print(escola_turma.shape)

escola_turma.head()

(214192, 13)


,CO_ENTIDADE,CO_MUNICIPIO,CO_UF,QT_DOC_BAS,QT_DOC_BAS_FEM,QT_DOC_BAS_MASC,QT_GESTORES,QT_TUR_BAS,QT_TUR_INF,QT_TUR_FUND,QT_TUR_MED,QT_TUR_EJA,QT_TUR_PROF
0,11022558,1100015,11,1.0,0.0,1.0,1.0,4.0,0.0,4.0,0.0,0.0,0.0
1,11024275,1100015,11,9.0,5.0,4.0,1.0,6.0,0.0,0.0,0.0,6.0,0.0
2,11024291,1100015,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11024666,1100015,11,13.0,10.0,3.0,1.0,11.0,2.0,9.0,0.0,0.0,0.0
4,11024682,1100015,11,44.0,24.0,20.0,1.0,37.0,0.0,9.0,28.0,0.0,5.0


In [29]:
escola_turma.to_parquet(
    SILVER / "escola_turma_enriquecida.parquet",
    index=False
)

print("Tabela escola_turma_enriquecida salva!")

Tabela escola_turma_enriquecida salva!


In [30]:
print(len(matricula.columns))
print(matricula.columns.tolist())

237
['NU_ANO_CENSO', 'CO_ENTIDADE', 'QT_MAT_BAS', 'QT_MAT_INF', 'QT_MAT_INF_CRE', 'QT_MAT_INF_PRE', 'QT_MAT_FUND', 'QT_MAT_FUND_AI', 'QT_MAT_FUND_AI_1', 'QT_MAT_FUND_AI_2', 'QT_MAT_FUND_AI_3', 'QT_MAT_FUND_AI_4', 'QT_MAT_FUND_AI_5', 'QT_MAT_FUND_AF', 'QT_MAT_FUND_AF_6', 'QT_MAT_FUND_AF_7', 'QT_MAT_FUND_AF_8', 'QT_MAT_FUND_AF_9', 'QT_MAT_MED', 'QT_MAT_MED_PROP', 'QT_MAT_MED_PROP_1', 'QT_MAT_MED_PROP_2', 'QT_MAT_MED_PROP_3', 'QT_MAT_MED_PROP_4', 'QT_MAT_MED_PROP_NS', 'QT_MAT_MED_IFTP_CT', 'QT_MAT_MED_IFTP_CT_1', 'QT_MAT_MED_IFTP_CT_2', 'QT_MAT_MED_IFTP_CT_3', 'QT_MAT_MED_IFTP_CT_4', 'QT_MAT_MED_IFTP_CT_NS', 'QT_MAT_MED_IFTP_QP', 'QT_MAT_MED_IFTP_QP_1', 'QT_MAT_MED_IFTP_QP_2', 'QT_MAT_MED_IFTP_QP_3', 'QT_MAT_MED_IFTP_QP_4', 'QT_MAT_MED_IFTP_QP_NS', 'QT_MAT_MED_NM', 'QT_MAT_MED_NM_1', 'QT_MAT_MED_NM_2', 'QT_MAT_MED_NM_3', 'QT_MAT_MED_NM_4', 'QT_MAT_MED_IFA', 'QT_MAT_MED_IFA_LING', 'QT_MAT_MED_IFA_LING_MT', 'QT_MAT_MED_IFA_LING_OTME', 'QT_MAT_MED_IFA_LING_OE', 'QT_MAT_MED_IFA_MATE', 'QT_MAT